# అధ్యాయము 7: చాట్ అప్లికేషన్లు నిర్మించడం
## అజ్యూర్ OpenAI API త్వరితప్రారंभం


## అవలోకనం
ఈ నోట్బుక్ [Azure OpenAI నమూనాలు రిపాజిటరీ](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) నుండి సరిపోయినది, ఇందులో [OpenAI](notebook-openai.ipynb) సేవను కూడా యాక్సెస్ చేసే నోట్బుక్లు ఉన్నాయి.

Python OpenAI API కొన్ని సవరణలతో Azure OpenAI తో కూడ పనిచేస్తుంది. ఇక్కడ తేడాల గురించి మరింత తెలుసుకోండి: [Python తో OpenAI మరియు Azure OpenAI ఎండ్పాయింట్ల మధ్య మార్చడం ఎలా](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

మరిన్ని వేగవంతమైన ఉదాహరణల కోసం అధికారిక [Azure OpenAI క్విక్‌స్టార్ట్ డాక్యుమెంటేషన్](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) చూడండి


## విషయం సూచిక  

[సారాంశం](#overview)  
[ఆజ్యూర్ ఓపెన్‌ఎఐ సేవతో ప్రారంభించడం](#getting-started-with-azure-openai-service)  
[మీ తొలి ప్రాంప్ట్ ని నిర్మించండి](#build-your-first-prompt)  

[వినియోగ కేసులు](#use-cases)    
[1. పాఠ్యాన్ని సంగ్రహించడం](#summarize-text)  
[2. పాఠ్యాన్ని వర్గీకరించడం](#classify-text)  
[3. కొత్త ఉత్పత్తి పేర్లను రూపొందించడం](#generate-new-product-names)  
[4. వర్గీకరించేవారి సన్నాహంకు ఫైన్భాగం](#fine-tune-a-classifier)  
[5. ఎంబెడ్డింగ్స్](#embeddings)

[సూచనలు](#references)


### Azure OpenAI సర్వీస్‌తో ప్రారంభించటం  

కొత్త ఖాతాదారులు Azure OpenAI సర్వీస్‌కు [ప్రవేశం కోసం దరఖాస్తు చేయాలి](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst).  
ఆమోదం పూర్తయిన తర్వాత, ఖాతాదారులు Azure పోర్టల్‌లో లాగిన్ అవ్వటం, ఒక Azure OpenAI సర్వీస్ వనరు సృష్టించడం, మరియు స్టూడియో ద్వారా మోడళ్లతో ప్రయోగం చేయడం ప్రారంభించవచ్చు  

[త్వరగా ప్రారంభించడానికి గొప్ప వనరు](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### మీ మొట్టమొదటి ప్రాంప్ట్‌ను రూపొందించండి  
ఈ చిన్న వ్యాయామం ఒక సాదారణ "సారాంశం" పనికి OpenAI మోడల్‌కు ప్రాంప్ట్‌లు సమర్పించడానికి ప్రాథమిక పరిచయాన్ని అందిస్తుంది.


**దశలు**:  
1. మీ పాథాన్ వాతావరణంలో OpenAI లైబ్రరీని ఇన్‌స్టాల్ చేయండి  
2. ప్రామాణిక సహాయక లైబ్రరీలను లోడ్ చేసి, మీరు సృష్టించిన OpenAI సేవకు మీ సాధారణ OpenAI భద్రతా సర్టిఫికెట్లను సెట్ చేయండి  
3. మీ పనికి ఒక మోడల్‌ను ఎంచుకోండి  
4. ఆ మోడల్ కోసం ఒక సులభమైన ప్రాంప్ట్‌ను రూపొందించండి  
5. మీ అభ్యర్థనను మోడల్ APIకి సమర్పించండి!


### 1. OpenAI ని ఇన్‌స్టాల్ చేయండి


  > [!NOTE] ఈ దశ అవసరం లేదు, మీరు ఈ నోట్‌బుక్‌ను కోడ్స్పేసెస్‌లో లేదా డెవ్‌కంటైనర్‌లో నడిపిస్తున్నట్లయితే


In [ ]:
%pip install openai python-dotenv

### 2. సహాయక లైబ్రరీలను దిగుమతి చేసుకుని ప్రమాణపత్రాలను సృష్టించండి


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. సరైన మోడల్‌ను కనుగొనడం  
GPT-4o మరియు GPT-4o మిని వంటి మోడల్స్ సహజ భాషను అర్థం చేసుకొని ఉత్పత్తి చేయగలవు. Microsoft Foundry లోని Azure OpenAI వివిధ విధుల కోసం గుర్తింపు మరియు వేగం వద్ద విభిన్న శక్తి స్థాయులతో వివిధ మోడల్ సామర్ధ్యాలను అందిస్తుంది. 

[Microsoft Foundryలోని Azure OpenAI మోడల్స్](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. ప్రాంప్ట్ డిజైన్  

"విస్తృత పాఠ్యాలపై ఈ అంచనా లోపాలను తగ్గించడానికి శిక్షణ పొందడం ద్వారా, పెద్ద భాషా నమూనాలు ఈ అంచనాల కోసం ఉపయోగకరమైన అభిప్రాయాలను నేర్చుకుంటాయి. ఉదాహరణకు, అవి ఈ క్రింది అభిప్రాయాలను నేర్చుకుంటాయి"(1):

* ఎలా స్పెల్ చేయాలి
* వ్యాకరణం ఎలా పనిచేస్తుంది
* ఎలా పారాఫ్రేస్ చేయాలి
* ఎలా ప్రశ్నలకు జవాబులు ఇవ్వాలి
* ఎలా సంభాషణ నిర్వహించాలి
* అనేక భాషల్లో ఎలా రాయాలి
* ఎలా కోడ్ చేయాలి
* తదితరాలు

#### పెద్ద భాషా నమూనాను ఎలా నియంత్రించాలి  
"పెద్ద భాషా నమూనాకు అందించే ఇన్పుట్లు అన్నిటిలో, అత్యంత ప్రభావవంతమైనది టెక్స్ట్ ప్రాంప్ట్(1).

పెద్ద భాషా నమూనాలను కొన్ని రకాలుగా ప్రాంప్ట్ చేయవచ్చు:

- సూచన: మీరు కావలసినదాన్ని నమూనాకు చెప్పండి
- పూర్తి చేయడం: మీరు కావలసినదాని ఆరంభాన్ని నమూనా పూర్తి చేయించాలని ప్రేరేపించండి
- ప్రదర్శన: మీరు కావలసినదాన్ని నమూనాకు చూపించండి, ఈ రెండు లో ఒకటి లేక రెండు కూడా:
- ప్రాంప్ట్‌లో కొంతమంది ఉదాహరణలు
- మెలకువ-శిక్షణ డేటా సెట్లో వందల కొద్దీ లేదా వేలాది ఉదాహరణలు



#### ప్రాంప్ట్‌లను సృష్టించడానికి మూడు ప్రాథమిక మార్గదర్శకాలు ఉన్నాయి:

**ప్రదర్శించండి మరియు చెప్పండి**. మీరు ఏమి కావాలో స్పష్టంగా చెప్పండి, సూచనల ద్వారా, ఉదాహరణల ద్వారా లేదా వీటిద్దరాల సంయోగంతో. మీరు నమూనా ఒక పదజాలాన్ని అక్షరమాల క్రమంలో ర్యాంక్ చేయాలని లేదా ఒక పేరాగ్రాఫ్‌ను భావజాలం ఆధారంగా వర్గీకరించాలనుకుంటే, అది కావాలని స్పష్టంగా చూపించండి.

**నాణ్యత గల డేటాను అందించండి**. మీరు క్లాసిఫయర్‌ని నిర్మించడానికి ప్రయత్నిస్తున్నారా లేదా నమూనాను ఒక నమూనాను అనుసరించించాలని కోరుతున్నారా, సరిపడిన ఉదాహరణలు ఉండాలి. మీ ఉదాహరణలను సరిచూసుకోండి — నమూనా సాధారణ స్పెల్లింగ్ తప్పిదాలను సగముగా గుర్తించి ప్రతిస్పందన ఇచ్చినా, ఇది ఉద్దేశపూర్వకమని అనుకుంటూ ప్రతిస్పందన లేదా ఫలితంపై ప్రభావం చూపవచ్చు.

**మీ సెట్టింగులను తనిఖీ చేయండి.** టెంపరేచర్ మరియు top_p సెట్టింగులు ప్రతిస్పందన సృష్టించడంలో నమూనా ఎంత నిర్ణాయకమో నియంత్రిస్తాయి. మీరు ఇక్కడ ఒక్క సరైన సమాధానం మాత్రమే ఉన్న ప్రశ్న అడుగుతున్నట్లైతే, ఈ విలువలను తక్కువగా సెట్ చేయాలి. మీరు విభిన్నమైన ప్రతిస్పందనలు కోరుకుంటే, వీటిని ఎక్కువగా ఉంచవచ్చు. ఈ సెట్టింగులతో ప్రజలు చేసే ప్రధాన తప్పు వాటిని "తేరుకైన" లేదా "సృజనాత్మక" నియంత్రణలుగా భావించడం.


మూలం: https://learn.microsoft.com/azure/ai-foundry/openai/overview


చిత్రం మీ మొదటి టెక్స్ట్ ప్రాంప్ట్‌ను సృష్టిస్తోంది!


### 5. దాఖలు చేయండి!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### అదే కాల్‌ను మళ్లీ చేయండి, ఫలితాలు ఎలా సరిపోలుతాయో చూడండి?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## టెక్స్ట్‌ను సారాంశం చేయండి  
#### సవాలు  
టెక్స్ట్ ప్యాసేజీ చివరకి 'tl;dr:' ను జోడించడం ద్వారా టెక్స్ట్‌ను సారాంశం చేయండి. మోడల్ ఎలా అదనపు సూచనలు లేకుండా పలు పనులను ఎంతగా అర్థం చేసుకుంటుందో గమనించండి. మోడల్ ప్రవర్తనను మార్చడానికి మరియు మీరు పొందే సారాంశాన్ని అనుకూలీకరించడానికి tl;dr కన్నా మరింత వివరణాత్మక ప్రాంప్ట్‌ల‌తో ప్రయోగాలు చేయవచ్చు(3).  

ఇటీవల జరిగిన పరిశోధనలు పెద్ద వచన శ్రేణిపై ముందస్తుగా శిక్షణ పొందిన తరువాత నిర్దిష్ట పనిపై ఫైన్-ట్యూనింగ్ చేసి NLP పనులు మరియు బెంచ్‌మార్క్‌లపై గణనీయమైన లాభాలను చూపించాయి. సాధారణంగా ఆర్కిటెక్చర్‌లో పనిపట్ల ఉపేక్షతో ఉండప్పటికీ, ఈ పద్ధతి వేల లేదా దశల సెంకిళ్ల ఉదాహరణలతో కూడిన పనిపట్ల ప్రత్యేక ఫైన్-ట్యూనింగ్ డేటాసెట్‌లను అవసరం చేస్తుంది. తులనాత్మకంగా, మానవులు సాధారణంగా కొద్ది ఉదాహరణలు లేదా సాదారణ సూచనల ద్వారా కొత్త భాషా పనిని నిర్వహించగలరు - ఇది ప్రస్తుత NLP వ్యవస్థలు ఇంకా ఎక్కువగా ఎదుర్కొంటున్న సమస్య. ఇక్కడ మేము భాషా మోడళ్లను భారీగా పెంచడం ద్వారా పనిపట్ల ఉపేక్షతో ఉండే, కొద్ది ఉదాహరణల పనితీరును మెరుగుపరచగలమని, కొన్ని సార్లు పూర్వ ప్రముఖ ఫైన్-ట్యూనింగ్ విధానాలతో కూడా పోటీ పడగలమని చూపుతున్నాము.  



సారాంశం  


# అనేక ఉపయోగాల కోసం వ్యాయామాలు  
1. పాఠాన్ని సారాంశం చేయండి  
2. పాఠాన్ని వర్గీకరించండి  
3. కొత్త ఉత్పత్తి పేర్లను రూపొందించండి
4. ఎంబెడ్డింగ్స్
5. ఒక వర్గీకర్తను మంచి విధంగా శిక్షణ ఇవ్వండి


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## టెక్స్ట్ వర్గీకరణ  
#### సవాలు  
నిర్దిష్ట సమయంలో ఇచ్చే వర్గాలకే వస్తువులను వర్గీకరించండి. దిగువ ఉదాహరణలో, మేము వర్గాలు మరియు వర్గీకరించవలసిన టెక్స్ట్‌ను(prompt(*playground_reference)) అందిస్తున్నాము. 

కస్టమర్ విచారణ: హలో, నా ల్యాప్‌టాప్ కీబోర్డు లోని ఒక కీలే ఇటీవల దెబ్బతిన్నది మరియు నాకు ఒక బదిలీ అవసరమవుతుంది:

వర్గీకరించిన వర్గం:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## కొత్త ఉత్పత్తి పేర్లు సృష్టించండి
#### సవాలు
ఉదాహరణ పదాల నుండి ఉత్పత్తి పేర్లను సృష్టించండి. ఇక్కడ మనం పేరు సృష్టించబోయే ఉత్పత్తి గురించి సమాచారం ప్రాంప్ట్‌లో చేర్చాము. మనకు కావలసిన నమూనాను చూపించడానికి సమానమైన ఒక ఉదాహరణ కూడా అందిస్తాము. రాండమ్నెస్ పెంచడానికి మరియు మరిన్ని ఆవిష్కరణాత్మక ప్రతిస్పందనలకు టెంపరేచర్ విలువ మామూలు కంటే ఎక్కువ పెట్టాము.

ఉత్పత్తి వివరణ: ఒక హోమ్ మిల్క్‌షేక్ మేకర్
సీడ్ పదాలు: వేగవంతమైన, ఆరోగ్యకరం, సంకీర్ణమైన.
ఉత్పత్తి పేర్లు: HomeShaker, Fit Shaker, QuickShake, Shake Maker

ఉత్పత్తి వివరణ: ఏ వ foot సైజుకి సరిపోయే షూ జంట.
సీడ్ పదాలు: అనుకూలమైన, సరిపోయే, ఒమ్మని-ఫిట్.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## ఎంబెడ్డింగ్స్
ఈ విభాగం ఎంబెడ్డింగ్స్‌ను ఎలా పొందాలో, మరియు పదాలు, వాక్యాలు, మరియు డాక్యుమెంట్ల మధ్య సమానతలను ఎలా కనుగొనాలో చూపిస్తుంది. తర్వాతి నోట్‌బుక్లు నడపడానికి మీరు `text-embedding-ada-002` ను బేస్ మోడల్‌గా ఉపయోగించే ఒక మోడల్‌ను డిప్లాయ్ చేసి, .env ఫైల్‌లో `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` వేరియబుల్ ద్వారా తన డిప్లాయ్‌మెంట్ పేరును సెట్ చేయాలి.


### మోడల్ వర్గీకరణ - ఒక ఎంబెడ్డింగ్ మోడల్ ఎంచుకోవడం

**మోడల్ వర్గీకరణ**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (ఎంబెడ్డింగ్స్ కుటుంబం)  
{capability} --> ada             (ఇతర ప్లే మోడల్స్ 2024 లో రిటైర్ చేయబడతాయి)  
{input-type} --> n/a             (శోధన మోడల్స్ కోసం మాత్రమే పేర్కొనబడింది)  
{identifier} --> 002             (సంస్కరణ 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] ఈ సన్నివేశాన్ని Codespaces లో లేదా Devcontainer లో నడుపుతున్నట్లయితే ఈ క్రింది దశ అవసరం లేదు


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## cnn డైలీ న్యూస్ డేటాసెట్ నుండి వ్యాసం పోలిక
మూలం: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# సూచనలు  
- [Microsoft Foundry - Azure OpenAI Models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# మరింత సహాయం కోసం  
[OpenAI వ్యాపారీకరణ బృందం](AzureOpenAITeam@microsoft.com)


# సహకారులు
* లూయిస్ లి  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
